In [11]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

In [12]:
# 1. Load Data
train = pd.read_csv('public/train.csv')
test = pd.read_csv('public/test.csv')

In [13]:
# 2. Preprocessing & Feature Engineering
def preprocess(df):
    # Handle pipe-separated strings (Count number of items)
    df['num_active_channels'] = df['active_channels'].fillna('').apply(lambda x: len(x.split('|')) if x != '' else 0)
    df['num_recent_promos'] = df['recent_promo_tools'].fillna('').apply(lambda x: len(x.split('|')) if x != '' else 0)
    
    # Interaction Features (The "Fit" signals)
    df['inv_efficiency'] = df['inventory_fill_rate'] * df['inventory_synergy']
    df['loyalty_match'] = df['repeat_buyer_rate'] * df['loyalty_synergy']
    df['margin_headroom'] = df['margin_rate'] - df['margin_penalty']
    
    # Categorical Encoding
    cat_cols = ['region', 'primary_category', 'seller_tier', 'promo_tool', 'tool_type', 'cost_tier']
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
    
    # Drop raw IDs and string fields not needed for the math
    drop_cols = ['candidate_id', 'query_id', 'snapshot_date', 'active_channels', 'recent_promo_tools']
    return df.drop(columns=[c for c in drop_cols if c in df.columns])

# Prepare datasets
X_train_full = preprocess(train.drop(columns=['is_relevant']))
y_train_full = train['is_relevant']
X_test = preprocess(test)

In [14]:
# 3. Temporal Validation Split
# Use the last 20% of the timeline to validate (mimics the test set shift)
split_idx = int(len(train) * 0.8)
X_tr, X_val = X_train_full.iloc[:split_idx], X_train_full.iloc[split_idx:]
y_tr, y_val = y_train_full.iloc[:split_idx], y_train_full.iloc[split_idx:]

# Define group sizes (Every query has 8 tools)
groups_tr = np.full(len(X_tr) // 8, 8)
groups_val = np.full(len(X_val) // 8, 8)
groups_test = np.full(len(X_test) // 8, 8)

In [15]:
# 4. Train LightGBM Ranker
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    label_gain=[0, 1], # is_relevant is binary (0 or 1)
    lambdarank_truncation_level=3, # Optimize specifically for NDCG@3
    learning_rate=0.05,
    n_estimators=1000,
    importance_type='gain'
)
features = [
    'region', 'primary_category', 'seller_tier', 'week_of_year', 'month', 'quarter',
    'seller_tenure_days', 'orders_30d', 'avg_order_value', 
    'margin_rate', 'listing_views_30d', 'conversion_rate', 'ad_spend_30d', 
    'search_visibility_score', 'inventory_fill_rate', 'stockout_rate_30d', 
    'repeat_buyer_rate', 'seller_rating', 'cashback_budget_score', 
    'marketing_readiness_score', 'promotion_fatigue_30d', 'is_cross_border', 
    'uses_fulfillment_service', 'holiday_campaign', 'month_end_push',
    'promo_tool', 'tool_type', 'discount_depth', 'visibility_boost', 
    'inventory_synergy', 'loyalty_synergy', 'margin_penalty', 'cross_border_fit', 
    'new_seller_fit', 'seasonal_fit', 'cost_tier', 'tool_recently_used',
    'num_active_channels', 'num_recent_promos', 'inv_efficiency', 
    'loyalty_match', 'margin_headroom'
]
ranker.fit(
    X_tr[features], y_tr,
    group=groups_tr,
    eval_set=[(X_val[features], y_val)],
    eval_group=[groups_val],
    eval_at=[3], # Monitor NDCG@3 during training
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.060089 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4543
[LightGBM] [Info] Number of data points in the train set: 168960, number of used features: 42
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[225]	valid_0's ndcg@3: 0.602112


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,1000
,subsample_for_bin,200000
,objective,'lambdarank'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [16]:
# 5. Generate Submission
test_scores = ranker.predict(X_test[features])
submission = pd.DataFrame({
    'candidate_id': test['candidate_id'],
    'score': test_scores
})

submission.to_csv('submission.csv', index=False)
print("Submission saved! Found 57,600 rows:", submission.shape[0] == 57600)

Submission saved! Found 57,600 rows: True
